# IDC Pancreas-CT: DICOM → NIfTI conversion QA

Run this notebook in Google Colab from Chrome. It queries IDC's public **Pancreas-CT** collection, downloads one CT series, converts it with `dcm2niix`, and compares the converted volume with an independent SimpleITK DICOM read. The GPU is not needed.

The collection has pancreas masks but is a **non-cancer control cohort**; this notebook is for conversion/anatomy QA, not lesion detection. Don't commit downloaded scans or generated volumes to GitHub.

In [ ]:
!apt-get -qq update && apt-get -qq install -y dcm2niix
!pip -q install idc-index SimpleITK numpy

## Find a manageable CT series

IDC's entire collection is very large. This query only displays a few candidate series; it does not download the whole collection. Choose one with a normal diagnostic description and a manageable size.

In [ ]:
from idc_index import IDCClient

client = IDCClient.client()
query = """
SELECT collection_id, PatientID, SeriesInstanceUID, SeriesDescription, series_size_MB
FROM index
WHERE collection_id = 'pancreas_ct' AND Modality = 'CT'
  AND series_size_MB BETWEEN 20 AND 1000
  AND lower(coalesce(SeriesDescription, '')) NOT LIKE '%scout%'
  AND lower(coalesce(SeriesDescription, '')) NOT LIKE '%localizer%'
ORDER BY series_size_MB ASC
LIMIT 12
"""
candidates = client.sql_query(query)
display(candidates)

In [ ]:
# Enter the row number (0-based) for one diagnostic CT series shown above.
choice = int(input('Candidate row number: '))
selected = candidates.iloc[choice]
series_uid = selected['SeriesInstanceUID']
print('Selected series:', series_uid)
print('Description:', selected['SeriesDescription'])
print('Approximate size (MB):', selected['series_size_MB'])

## Download just that series

The download is made into a UID-named directory so the conversion command can target exactly one series.

In [ ]:
from pathlib import Path

data_root = Path('/content/data/idc')
data_root.mkdir(parents=True, exist_ok=True)
client.download_dicom_series(
    seriesInstanceUID=series_uid,
    downloadDir=str(data_root),
    dirTemplate='%SeriesInstanceUID',
    quiet=False,
    show_progress_bar=True,
)
dicom_dir = data_root / series_uid
print('DICOM folder:', dicom_dir)
print('Files:', sum(1 for p in dicom_dir.iterdir() if p.is_file()))

## Convert, then inspect QA report

First fork or push the starter project to GitHub, then replace `YOUR_GITHUB_USERNAME` below with the account name.

In [ ]:
!git clone -q https://github.com/YOUR_GITHUB_USERNAME/pancreas-conversion-qa.git /content/pancreas-conversion-qa
%cd /content/pancreas-conversion-qa
!pip -q install -e .
!ctqa run "{dicom_dir}" --out /content/outputs

The report is written to `/content/outputs/qa-report.json`. A geometry mismatch or low physical overlap stops the run. Review converter logs and the JSON before relying on the output. This simple check is not a substitute for a visual review or an independent validation study.

## Data context

IDC's Pancreas-CT collection contains contrast-enhanced abdominal CT from 80 subjects and accompanying pancreas segmentations; the collection description says these are healthy controls or patients without pancreatic cancer lesions. The pancreas mask could support a later segmentation-alignment check. For actual lesion segmentation, use the separate MSD Task07 Pancreas dataset, which is already distributed as NIfTI.